# NINAAD WAGLE | BTECH AI SEM V | I065 B2 | NLP LAB 5

# Task a : POS tagging and Named Entity Recognition

**POS tagging** labels every word with its part of speech - noun, verb, adjective
and so on. The tag depends on how the word is used, so the same word can get
different tags in different sentences.

**Named Entity Recognition (NER)** goes one step further and picks out the groups
of words that name a real thing - a person, an organisation, a place, a date - and
says which kind it is.

The same text is passed through NLTK and then through spaCy so the two outputs can
be compared.

In [1]:
text = ("Barack Obama was born in Hawaii and later served as the President of the United States. "
        "He studied law at Harvard University and worked in Chicago for many years. "
        "In 2009 he received the Nobel Peace Prize in Oslo.")

print(text)

Barack Obama was born in Hawaii and later served as the President of the United States. He studied law at Harvard University and worked in Chicago for many years. In 2009 he received the Nobel Peace Prize in Oslo.


## i : Using NLTK

NLTK works on a plain list of tokens. `word_tokenize` splits the text into words and
`pos_tag` gives each token a **Penn Treebank** tag, for example `NNP` for a proper
noun, `VBD` for a past tense verb and `IN` for a preposition.

In [2]:
import pandas as pd
from nltk import word_tokenize, pos_tag, ne_chunk

tokens = word_tokenize(text)
nltk_tags = pos_tag(tokens)

print("total tokens:", len(tokens))
pd.DataFrame(nltk_tags, columns=["token", "pos tag"]).head(15)

total tokens: 42


,token,pos tag
0,Barack,NNP
1,Obama,NNP
2,was,VBD
3,born,VBN
4,in,IN
5,Hawaii,NNP
6,and,CC
7,later,RB
8,served,VBD
9,as,IN


`ne_chunk` takes those tagged tokens and returns a tree. The plain tokens stay as
leaves, while a named entity becomes a small subtree whose label is the entity type,
so the entities are exactly the parts of the tree that have a `label`.

In [3]:
tree = ne_chunk(nltk_tags)

nltk_entities = [(" ".join(word for word, tag in subtree), subtree.label())
                 for subtree in tree if hasattr(subtree, "label")]

pd.DataFrame(nltk_entities, columns=["entity", "label"])

,entity,label
0,Barack,PERSON
1,Obama,PERSON
2,Hawaii,GPE
3,United States,GPE
4,Harvard University,ORGANIZATION
5,Chicago,GPE
6,Nobel Peace Prize,ORGANIZATION
7,Oslo,GPE


## ii : Using spaCy

spaCy runs the whole pipeline in one call. `nlp(text)` returns a `Doc` in which every
token already carries its tags: `pos_` is the simple universal tag (`PROPN`, `VERB`)
and `tag_` is the detailed Penn Treebank tag that NLTK also uses.

In [4]:
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp(text)

print("total tokens:", len(doc))
pd.DataFrame([(token.text, token.pos_, token.tag_) for token in doc],
             columns=["token", "pos", "tag"]).head(15)

total tokens: 42


,token,pos,tag
0,Barack,PROPN,NNP
1,Obama,PROPN,NNP
2,was,AUX,VBD
3,born,VERB,VBN
4,in,ADP,IN
5,Hawaii,PROPN,NNP
6,and,CCONJ,CC
7,later,ADV,RB
8,served,VERB,VBD
9,as,ADP,IN


The named entities are already stored in `doc.ents`, so nothing extra has to be run.
`spacy.explain` turns the short label into words.

In [5]:
pd.DataFrame([(ent.text, ent.label_, spacy.explain(ent.label_)) for ent in doc.ents],
             columns=["entity", "label", "meaning"])

,entity,label,meaning
0,Barack Obama,PERSON,"People, including fictional"
1,Hawaii,GPE,"Countries, cities, states"
2,the United States,GPE,"Countries, cities, states"
3,Harvard University,ORG,"Companies, agencies, institutions, etc."
4,Chicago,GPE,"Countries, cities, states"
5,many years,DATE,Absolute or relative dates or periods
6,2009,DATE,Absolute or relative dates or periods
7,the Nobel Peace Prize,WORK_OF_ART,"Titles of books, songs, etc."
8,Oslo,GPE,"Countries, cities, states"


Both tools agree on the parts of speech, because spaCy's `tag_` uses the same Penn
Treebank set as NLTK.

The difference shows up in NER. NLTK splits *Barack Obama* into two separate
`PERSON` entities, while spaCy keeps the full name together. NLTK also has only a
few broad labels, so it calls *Nobel Peace Prize* an `ORGANIZATION` and finds no
dates at all. spaCy has many more labels and picks up *2009* and *many years* as
`DATE` and the prize as `WORK_OF_ART`.

# Task b : Feature engineering on a real world review dataset

The dataset used here is **product_reviews_1**, which ships with NLTK. It holds 313
customer reviews written for five electronic products - a DVD player, two cameras,
an mp3 player and a mobile phone. Every review also carries the product features the
customer talked about along with an opinion score, so a sentiment label can be worked
out from those scores.

The `review` text is the predictor column. The three steps below clean it, then build
numeric features out of it, so that the reviews become a table a model could be
trained on.

In [6]:
import nltk
import pandas as pd
from nltk.corpus import product_reviews_1 as reviews

nltk.download("product_reviews_1", quiet=True)

rows = []
for fileid in reviews.fileids():
    if fileid == "README.txt":
        continue
    for review in reviews.reviews(fileid):
        text = " ".join(" ".join(sent) for sent in review.sents())
        score = sum(int(polarity) for _, polarity in review.features())
        rows.append((fileid.replace(".txt", ""), text, score))

df = pd.DataFrame(rows, columns=["product", "review", "score"])
df["sentiment"] = df["score"].apply(lambda s: "positive" if s > 0 else "negative" if s < 0 else "neutral")

print("reviews:", len(df), "| products:", df["product"].nunique())
print(df["sentiment"].value_counts().to_dict())
df[["product", "review", "sentiment"]].head()

reviews: 313 | products: 5
{'positive': 188, 'negative': 109, 'neutral': 16}


,product,review,sentiment
0,Apex_AD2600_Progressive_scan_DVD player,"repost from january 13 , 2004 with a better fi...",positive
1,Apex_AD2600_Progressive_scan_DVD player,i ' ve owned 6 or 7 dvd players since 1998 . t...,positive
2,Apex_AD2600_Progressive_scan_DVD player,many of our disney movies do n ' t play on thi...,negative
3,Apex_AD2600_Progressive_scan_DVD player,player has a problem with dual - layer dvd ' s...,negative
4,Apex_AD2600_Progressive_scan_DVD player,"for the first few weeks , this player was ever...",negative


## i : Removing the noise

Noise is everything in the text that does not help tell one review from another.
Here that means the case of a letter, the punctuation and the digits, and the
**stopwords** - very common words like *the*, *is* and *and* that appear in every
review.

The regular expression keeps only letters and spaces, and the stopword list from
NLTK removes the rest.

In [7]:
import re
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def clean(text):
    text = re.sub(r"[^a-z\s]", " ", text.lower())
    return " ".join(word for word in text.split() if word not in stop_words)

df["clean_review"] = df["review"].apply(clean)

print("before :", df["review"][0][:120])
print("after  :", df["clean_review"][0][:120])
df[["review", "clean_review"]].head()

before : repost from january 13 , 2004 with a better fit title . does your apex dvd player only play dvd audio without video ? or
after  : repost january better fit title apex dvd player play dvd audio without video play audio video scrolling black white try 


,review,clean_review
0,"repost from january 13 , 2004 with a better fi...",repost january better fit title apex dvd playe...
1,i ' ve owned 6 or 7 dvd players since 1998 . t...,owned dvd players since far nicest one many wa...
2,many of our disney movies do n ' t play on thi...,many disney movies n play dvd player spoke rep...
3,player has a problem with dual - layer dvd ' s...,player problem dual layer dvd alias season sea...
4,"for the first few weeks , this player was ever...",first weeks player everything expected afforda...


## ii : Length based features

These four numbers describe how much a customer wrote and how varied the writing is.
`word_count` is how many words are left after cleaning, `total_word_length` is the
number of letters in all of those words, `unique_words` counts how many of them are
different, and `bigrams` counts the distinct pairs of neighbouring words.

A long review with few unique words is repetitive, while one where the unique count
is close to the word count says something new in almost every word.

In [8]:
from nltk import bigrams

words = df["clean_review"].str.split()

df["word_count"] = words.apply(len)
df["total_word_length"] = df["clean_review"].str.replace(" ", "").str.len()
df["unique_words"] = words.apply(lambda w: len(set(w)))
df["bigrams"] = words.apply(lambda w: len(set(bigrams(w))))

df[["clean_review", "word_count", "total_word_length", "unique_words", "bigrams"]].head()

,clean_review,word_count,total_word_length,unique_words,bigrams
0,repost january better fit title apex dvd playe...,191,1051,122,169
1,owned dvd players since far nicest one many wa...,181,897,143,175
2,many disney movies n play dvd player spoke rep...,23,103,21,21
3,player problem dual layer dvd alias season sea...,55,321,43,53
4,first weeks player everything expected afforda...,66,360,59,65


## iii : POS tag and named entity tag features

The tagger from Task a is now run over every review to turn the grammar into numbers -
how many nouns, proper nouns, verbs, adjectives and adverbs a review uses, how many
named entities it mentions, and which entity types those are.

The **original** review is tagged and not the cleaned one, because removing the
stopwords breaks the sentence structure that spaCy needs to decide a tag.

In [9]:
import spacy
from collections import Counter

nlp = spacy.load("en_core_web_sm")
docs = list(nlp.pipe(df["review"]))

for pos in ["NOUN", "PROPN", "VERB", "ADJ", "ADV"]:
    df[pos.lower() + "_count"] = [sum(token.pos_ == pos for token in doc) for doc in docs]

df["entity_count"] = [len(doc.ents) for doc in docs]
df["entity_tags"] = [", ".join(sorted({ent.label_ for ent in doc.ents})) for doc in docs]

print("most common entity tags:", Counter(ent.label_ for doc in docs for ent in doc.ents).most_common(5))
df[["noun_count", "propn_count", "verb_count", "adj_count", "adv_count",
    "entity_count", "entity_tags"]].head()

most common entity tags: [('CARDINAL', 852), ('DATE', 363), ('ORG', 157), ('ORDINAL', 141), ('PERSON', 131)]


,noun_count,propn_count,verb_count,adj_count,adv_count,entity_count,entity_tags
0,78,24,54,20,19,7,"DATE, ORDINAL, ORG, PERSON, TIME"
1,71,13,48,40,26,17,"CARDINAL, DATE, ORDINAL, ORG, QUANTITY"
2,13,0,5,4,0,1,DATE
3,29,3,16,8,4,1,DATE
4,26,3,21,11,10,4,"CARDINAL, DATE, MONEY, ORG"


In [10]:
df.drop(columns=["review", "clean_review", "score"]).head()

,product,sentiment,word_count,total_word_length,unique_words,bigrams,noun_count,propn_count,verb_count,adj_count,adv_count,entity_count,entity_tags
0,Apex_AD2600_Progressive_scan_DVD player,positive,191,1051,122,169,78,24,54,20,19,7,"DATE, ORDINAL, ORG, PERSON, TIME"
1,Apex_AD2600_Progressive_scan_DVD player,positive,181,897,143,175,71,13,48,40,26,17,"CARDINAL, DATE, ORDINAL, ORG, QUANTITY"
2,Apex_AD2600_Progressive_scan_DVD player,negative,23,103,21,21,13,0,5,4,0,1,DATE
3,Apex_AD2600_Progressive_scan_DVD player,negative,55,321,43,53,29,3,16,8,4,1,DATE
4,Apex_AD2600_Progressive_scan_DVD player,negative,66,360,59,65,26,3,21,11,10,4,"CARDINAL, DATE, MONEY, ORG"


The raw text column has now become thirteen numeric and categorical features.

The counts line up with what the reviews look like. Nouns are the largest group
because customers keep naming parts of the product, and adjectives come next since a
review is mostly opinion. `CARDINAL` and `DATE` are the entity tags that appear most
often - people quote prices, model numbers, battery hours and purchase dates far more
than they name persons or places.